In [1]:
offline_path = "/kaggle/input/datasets/quyennv1912/20-newspaper-offline"

In [2]:
from sklearn.datasets import fetch_20newsgroups
import pandas as pd

dataset = fetch_20newsgroups(subset="train", data_home=offline_path,)

df = pd.DataFrame({
    'target': dataset.target,
    'target_name': [dataset.target_names[i] for i in dataset.target]
})

dis = df['target_name'].value_counts()

print(dis)

target_name
rec.sport.hockey            600
soc.religion.christian      599
rec.motorcycles             598
rec.sport.baseball          597
sci.crypt                   595
sci.med                     594
rec.autos                   594
sci.space                   593
comp.windows.x              593
comp.os.ms-windows.misc     591
sci.electronics             591
comp.sys.ibm.pc.hardware    590
misc.forsale                585
comp.graphics               584
comp.sys.mac.hardware       578
talk.politics.mideast       564
talk.politics.guns          546
alt.atheism                 480
talk.politics.misc          465
talk.religion.misc          377
Name: count, dtype: int64


In [3]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

remove = ('headers', 'footers', 'quotes')

train = fetch_20newsgroups(subset="train", data_home=offline_path, remove=remove)
test = fetch_20newsgroups(subset="test", data_home=offline_path, remove=remove)

pipe = make_pipeline(
    # min_df quy định số lần xuất hiện ít nhất của 1 từ trong văn bản
    # ngram_range quy định các từ phải được gom nhóm
    TfidfVectorizer(sublinear_tf=True, stop_words="english", min_df=2, ngram_range=(1, 2)),
    LogisticRegression(max_iter=5000)
)

grid = GridSearchCV(
    pipe, 
    {"logisticregression__C": [5, 10, 20, 30, 40, 50, 100]}, 
    cv=5, 
    scoring="f1_macro", 
    n_jobs=-1
)
grid.fit(train.data, train.target)

print(grid.best_params_)
print(classification_report(test.target, grid.predict(test.data), target_names=train.target_names))

{'logisticregression__C': 50}
                          precision    recall  f1-score   support

             alt.atheism       0.51      0.49      0.50       319
           comp.graphics       0.66      0.74      0.70       389
 comp.os.ms-windows.misc       0.64      0.62      0.63       394
comp.sys.ibm.pc.hardware       0.66      0.67      0.67       392
   comp.sys.mac.hardware       0.73      0.70      0.71       385
          comp.windows.x       0.85      0.70      0.77       395
            misc.forsale       0.78      0.80      0.79       390
               rec.autos       0.49      0.80      0.61       396
         rec.motorcycles       0.81      0.75      0.78       398
      rec.sport.baseball       0.80      0.82      0.81       397
        rec.sport.hockey       0.90      0.87      0.88       399
               sci.crypt       0.84      0.71      0.77       396
         sci.electronics       0.63      0.61      0.62       393
                 sci.med       0.79      0.77

In [7]:
import copy
import random
import re
import time
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# ===================== SIÊU THAM SỐ =====================
SEED = 42
MAX_VOCAB = 20000
MIN_FREQ = 3
MAX_LEN = 200
BATCH_SIZE = 32
EMBEDDING_DIM = 100
HIDDEN_DIM = 120
NUM_CLASSES = 20

DROPOUT = 0.5
LR = 1e-3
EPOCHS = 50
PATIENCE = 5

# --- Gradual unfreezing + discriminative LR ---
UNFREEZE_EPOCH = 5      # epoch (1-indexed) tại đó embedding được unfreeze
EMB_LR_FACTOR = 0.05    # LR của embedding = LR * factor sau khi unfreeze

PAD_ID, UNK_ID = 0, 1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===================== DỮ LIỆU & TIỀN XỬ LÝ =====================
remove = ('headers', 'footers', 'quotes')

train_full = fetch_20newsgroups(subset="train", remove=remove, data_home=offline_path)
test_data = fetch_20newsgroups(subset="test", remove=remove, data_home=offline_path)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_full.data, train_full.target, test_size=0.1, stratify=train_full.target, random_state=SEED
)

URL_RE = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE = re.compile(r'\S+@\S+')
NON_ALPHA_RE = re.compile(r'[^a-z\s]')

def tokenize(text):
    text = NON_ALPHA_RE.sub(" ", EMAIL_RE.sub(" ", URL_RE.sub(" ", text.lower())))
    return [w for w in text.split() if len(w) >= 3 and w not in ENGLISH_STOP_WORDS]

train_tokens = [tokenize(t) for t in train_texts]
val_tokens = [tokenize(t) for t in val_texts]
test_tokens = [tokenize(t) for t in test_data.data]

counter = Counter(w for tokens in train_tokens for w in tokens)
vocab = {"<PAD>": PAD_ID, "<UNK>": UNK_ID}
for word, freq in counter.most_common(MAX_VOCAB - len(vocab)):
    if freq < MIN_FREQ:
        break  # most_common() đã sắp giảm dần -> có thể dừng sớm
    vocab[word] = len(vocab)

class TextDataset(Dataset):
    def __init__(self, token_lists, labels, vocab):
        self.labels = labels
        self.samples = [torch.tensor([vocab.get(w, UNK_ID) for w in tokens[:MAX_LEN]], dtype=torch.long) for tokens in token_lists]
    def __len__(self): return len(self.samples)
    def __getitem__(self, index): return self.samples[index], torch.tensor(int(self.labels[index]), dtype=torch.long)

# Đóng gói dữ liệu cho DATALOADER
def text_collate_fn(batch):
    x_list, y_list, lengths = [], [], []
    for x, y in batch:
        x = torch.tensor([UNK_ID], dtype=torch.long) if len(x) == 0 else x
        x_list.append(x); y_list.append(y); lengths.append(len(x))
    return pad_sequence(x_list, batch_first=True, padding_value=PAD_ID), torch.stack(y_list), torch.tensor(lengths)

train_loader = DataLoader(TextDataset(train_tokens, train_labels, vocab), batch_size=BATCH_SIZE, shuffle=True, collate_fn=text_collate_fn)
val_loader = DataLoader(TextDataset(val_tokens, val_labels, vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=text_collate_fn)
test_loader = DataLoader(TextDataset(test_tokens, test_data.target, vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=text_collate_fn)

# ===================== GLOVE EMBEDDING =====================
glove_path = "/kaggle/input/datasets/quyennv1912/glove6b100dtxt/glove.6B.100d.txt"
embedding_matrix = np.zeros((len(vocab), EMBEDDING_DIM))
try:
    with open(glove_path, 'r', encoding='utf-8') as f:
        glove_dict = {line.split()[0]: np.asarray(line.split()[1:], "float32") for line in f}
    for word, idx in vocab.items():
        if idx != PAD_ID:
            # Nếu từ không có trong vocab thì sẽ khởi tạo ngấu nhiên theo phân phối chuẩn sclae = 0.6
            embedding_matrix[idx] = glove_dict.get(word, np.random.normal(scale=0.6, size=(EMBEDDING_DIM,)))
except FileNotFoundError:
    print("Không tìm thấy GloVe, khởi tạo ngẫu nhiên weights.")
    embedding_matrix = np.random.normal(scale=0.6, size=(len(vocab), EMBEDDING_DIM))

embedding_tensor = torch.tensor(embedding_matrix, dtype=torch.float32)

# ===================== MÔ HÌNH TỔNG QUÁT =====================
class TextModel(nn.Module):
    def __init__(self, rnn_type, hidden_dim, num_classes, dropout, pretrained_weights, emb_dropout=0.3):
        super().__init__()
        self.rnn_type = rnn_type.lower()
        # freeze=True chỉ là trạng thái BAN ĐẦU; sẽ unfreeze thủ công sau UNFREEZE_EPOCH
        self.embedding = nn.Embedding.from_pretrained(pretrained_weights, freeze=True, padding_idx=PAD_ID)
        self.emb_dropout = nn.Dropout(emb_dropout)

        rnn_classes = {'rnn': nn.RNN, 'lstm': nn.LSTM, 'gru': nn.GRU}
        self.rnn = rnn_classes[self.rnn_type](
            input_size=pretrained_weights.shape[1], hidden_size=hidden_dim,
            num_layers=1, batch_first=True, bidirectional=True
        )
        self.fc_dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 4, num_classes)

    def unfreeze_embedding(self):
        self.embedding.weight.requires_grad = True

    def forward(self, x, lengths):
        if self.training:
            drop_mask = (torch.rand(x.shape, device=x.device) < 0.1) & (x != PAD_ID)
            x = x.masked_fill(drop_mask, UNK_ID)

        encoded = self.emb_dropout(self.embedding(x))
        lengths = lengths.clamp(min=1)
        packed = pack_padded_sequence(encoded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out_packed, _ = self.rnn(packed)
        out, _ = pad_packed_sequence(out_packed, batch_first=True)

        lengths = lengths.to(out.device)
        mask = (torch.arange(out.size(1), device=out.device)[None, :] < lengths[:, None]).unsqueeze(-1)

        max_pool = out.masked_fill(~mask, -1e9).max(dim=1).values
        mean_pool = (out * mask).sum(dim=1) / lengths.unsqueeze(-1).float()

        return self.fc(self.fc_dropout(torch.cat([max_pool, mean_pool], dim=1)))

# ===================== OPTIMIZER (DISCRIMINATIVE LR) =====================
def build_optimizer(model, embedding_trainable):
    """
    - Trước khi unfreeze: chỉ đưa các tham số requires_grad=True (RNN + FC) vào optimizer.
    - Sau khi unfreeze: 2 param group - embedding LR nhỏ, phần còn lại LR chuẩn.
    """
    if not embedding_trainable:
        params = [p for p in model.parameters() if p.requires_grad]
        return torch.optim.AdamW(params, lr=LR, weight_decay=1e-2)

    return torch.optim.AdamW([
        # LR của phần embedding sẽ được nhân với EMB_LR_FACTOR
        {"params": model.embedding.parameters(), "lr": LR * EMB_LR_FACTOR},
        # LR của phần còn lại sẽ được giữ nguyên
        {"params": [p for n, p in model.named_parameters() if not n.startswith("embedding")], "lr": LR},
    ], weight_decay=1e-2)

# ===================== HÀM ĐÁNH GIÁ & HUẤN LUYỆN =====================
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, preds, targets = 0.0, [], []
    with torch.no_grad():
        for batch_x, batch_y, lengths in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x, lengths)
            total_loss += criterion(outputs, batch_y).item() * batch_y.size(0)
            preds.extend(outputs.argmax(dim=1).cpu().tolist())
            targets.extend(batch_y.cpu().tolist())

    acc = sum(p == t for p, t in zip(preds, targets)) / len(targets)
    return total_loss / len(targets), acc, preds, targets

def train_and_evaluate(model_name):
    print(f"\n{'='*20} BẮT ĐẦU HUẤN LUYỆN: {model_name.upper()} {'='*20}")
    model = TextModel(model_name, HIDDEN_DIM, NUM_CLASSES, DROPOUT, embedding_tensor).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    embedding_trainable = False
    optimizer = build_optimizer(model, embedding_trainable)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)

    best_val_loss, best_state, bad_epochs = float("inf"), None, 0

    start_time = time.time()

    for epoch in range(1, EPOCHS + 1):
        # ---- Gradual unfreezing: chuyển pha tại UNFREEZE_EPOCH ----
        if epoch == UNFREEZE_EPOCH and not embedding_trainable:
            model.unfreeze_embedding()
            embedding_trainable = True
            optimizer = build_optimizer(model, embedding_trainable)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)
            print(f"--> Epoch {epoch}: unfreeze embedding, LR embedding = {LR * EMB_LR_FACTOR:.2e}, LR còn lại = {LR:.2e}")

        model.train()
        total_loss = 0.0
        for batch_x, batch_y, lengths in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_x, lengths), batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)

        lrs = [g['lr'] for g in optimizer.param_groups]
        lr_str = "/".join(f"{lr:.2e}" for lr in lrs)
        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc*100:.2f}% | lr={lr_str}")

        if val_loss < best_val_loss:
            best_val_loss, best_state, bad_epochs = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("Early stopping. Mô hình không cải thiện thêm.")
                break

    end_time = time.time()
    elapsed_time = end_time - start_time
    mins = int(elapsed_time // 60)
    secs = int(elapsed_time % 60)
    print(f"\n=> Tổng thời gian huấn luyện ({model_name.upper()}): {mins} phút {secs} giây")

    model.load_state_dict(best_state)
    _, test_acc, preds, targets = evaluate(model, test_loader, criterion)
    print(f"Accuracy ({model_name.upper()}) trên tập test: {test_acc * 100:.2f}%\n")
    print(classification_report(targets, preds, target_names=test_data.target_names, digits=4, zero_division=0))

# ===================== CHẠY THỬ NGHIỆM =====================
print(f"Thiết bị sử dụng: {device}")
print(f"Kích thước từ điển: {len(vocab)}")

for architecture in ['rnn', 'lstm', 'gru']:
    train_and_evaluate(architecture)

Thiết bị sử dụng: cuda
Kích thước từ điển: 20000

==================== BẮT ĐẦU HUẤN LUYỆN: RNN ====================
Epoch 01 | train_loss=2.4229 | val_loss=2.0201 | val_acc=40.11% | lr=1.00e-03
Epoch 02 | train_loss=2.0794 | val_loss=1.8895 | val_acc=50.44% | lr=1.00e-03
Epoch 03 | train_loss=1.9948 | val_loss=1.8532 | val_acc=49.47% | lr=1.00e-03
Epoch 04 | train_loss=1.9469 | val_loss=1.7946 | val_acc=51.33% | lr=1.00e-03
--> Epoch 5: unfreeze embedding, LR embedding = 5.00e-05, LR còn lại = 1.00e-03
Epoch 05 | train_loss=1.9264 | val_loss=1.7521 | val_acc=55.12% | lr=5.00e-05/1.00e-03
Epoch 06 | train_loss=1.8823 | val_loss=1.6982 | val_acc=58.57% | lr=5.00e-05/1.00e-03
Epoch 07 | train_loss=1.8668 | val_loss=1.7335 | val_acc=57.07% | lr=5.00e-05/1.00e-03
Epoch 08 | train_loss=1.8293 | val_loss=1.6952 | val_acc=57.60% | lr=5.00e-05/1.00e-03
Epoch 09 | train_loss=1.8094 | val_loss=1.6912 | val_acc=58.30% | lr=5.00e-05/1.00e-03
Epoch 10 | train_loss=1.7933 | val_loss=1.6498 | val_acc=

## Vì sao TF-IDF + Logistic Regression tốt hơn RNN/LSTM/GRU?

Trên 20 Newsgroups, việc phân loại chủ đề gần như hoàn toàn dựa vào **sự xuất hiện của các từ khóa đặc trưng** (ví dụ "hockey", "encryption", "atheism"), chứ không phụ thuộc vào ngữ cảnh hay trật tự từ. Đây chính là "sân nhà" của TF-IDF: nó bắt trực tiếp tín hiệu từ vựng phân biệt lớp, kể cả các cụm từ hiếm nhờ `min_df` thấp và bigram.

Ngược lại, RNN/LSTM/GRU được thiết kế để mô hình hóa **phụ thuộc tuần tự** — một khả năng gần như thừa cho bài toán này — trong khi phải học từ một tập dữ liệu khá nhỏ (~11k mẫu cho 20 lớp). Với quy mô dữ liệu đó, mô hình sequence nhiều tham số dễ khó tối ưu hơn là mang lại lợi thế.


**1. Độ chính xác (Accuracy & F1-Score)**

* **GRU dẫn đầu (65.39%)**, bám sát ngay sau là LSTM (65.03%) và RNN lùi lại phía sau (64.05%).
* Dù RNN đã thu hẹp khoảng cách đáng kể khi được phép chạy nhiều epoch hơn, GRU và LSTM vẫn thể hiện sự vượt trội rõ rệt ở các chủ đề khó, dễ nhầm lẫn (tôn giáo, chính trị) nhờ cơ chế kiểm soát bộ nhớ giúp xử lý ngữ cảnh văn bản dài.

**2. Thời gian và Tốc độ hội tụ**

* **LSTM và GRU hội tụ cực nhanh:** LSTM chỉ mất 1 phút 21 giây (kích hoạt Early Stopping ở epoch 23), trong khi GRU mất 1 phút 27 giây (dừng ở epoch 25).
* **RNN chật vật và tốn thời gian nhất (2 phút 49 giây):** Do hiện tượng suy biến đạo hàm (vanishing gradient), RNN gặp khó khăn trong việc học và phải rà soát mòn mỏi tới 49 epoch mới chạm được ngưỡng bão hòa.

**3. Khả năng tổng quát hóa (Generalization)**

* **GRU và LSTM xuất sắc:** Nắm bắt đặc trưng nhanh, đạt đỉnh Validation sớm (chỉ cần khoảng nửa số epoch tối đa) và dừng đúng lúc nhờ Early Stopping, bảo toàn năng lực dự đoán trên tập test thực tế.
* **RNN kém hiệu quả nhất:** Dù cơ chế Early Stopping đã cứu RNN khỏi việc bị overfit nặng, nhưng việc phải loay hoay qua gần 50 vòng lặp cho thấy giới hạn của cấu trúc này. Nó phải bù đắp sự yếu kém về mặt kiến trúc bằng cách học lâu hơn rất nhiều, nhưng kết quả cuối cùng vẫn thấp nhất.